In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "figure_notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from eval import load as eval_load
from eval import passat

FIGURE_DIR = REPO_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 14,
    "figure.dpi": 120,
    "savefig.dpi": 300,
})


In [ ]:
DATASET = "humaneval"
DATASET_SIZE = 164
VARIANTS = ["temperature", "retok"]
LOW_TAIL_THRESHOLD = 0.1
HIGH_TAIL_THRESHOLD = 0.75

SIZE_MODELS = [
    "allenai_OLMo-2-0425-1B-Instruct",
    "allenai_OLMo-2-1124-7B-Instruct",
    "allenai_OLMo-2-1124-13B-Instruct",
    "allenai_OLMo-2-0325-32B-Instruct",
]

CHECKPOINT_MODELS = [
    "allenai_OLMo-2-1124-7B_step_1000",
    "allenai_OLMo-2-1124-7B_step_10000",
    "allenai_OLMo-2-1124-7B_step_30000",
    "allenai_OLMo-2-1124-7B_step_60000",
    "allenai_OLMo-2-1124-7B_step_101000",
    "allenai_OLMo-2-1124-7B_step_300000",
    "allenai_OLMo-2-1124-7B_step_500000",
    "allenai_OLMo-2-1124-7B_step_700000",
    "allenai_OLMo-2-1124-7B_step_900000",
]

POSTTRAIN_MODELS = [
    "allenai/OLMo-2-1124-7B",
    "allenai/OLMo-2-1124-7B-SFT",
    "allenai/OLMo-2-1124-7B-DPO",
    "allenai/OLMo-2-1124-7B-Instruct",
]

ORDERED_MODELS = CHECKPOINT_MODELS + POSTTRAIN_MODELS
LOAD_MODELS = SIZE_MODELS + [model_name for model_name in ORDERED_MODELS if model_name not in SIZE_MODELS]
MODEL_LABELS = {
    "allenai_OLMo-2-1124-7B_step_1000": "pre-training\nstep 1k",
    "allenai_OLMo-2-1124-7B_step_10000": "10k",
    "allenai_OLMo-2-1124-7B_step_30000": "30k",
    "allenai_OLMo-2-1124-7B_step_60000": "60k",
    "allenai_OLMo-2-1124-7B_step_101000": "101k",
    "allenai_OLMo-2-1124-7B_step_300000": "300k",
    "allenai_OLMo-2-1124-7B_step_500000": "500k",
    "allenai_OLMo-2-1124-7B_step_700000": "700k",
    "allenai_OLMo-2-1124-7B_step_900000": "900k",
    "allenai/OLMo-2-1124-7B": "Base",
    "allenai/OLMo-2-1124-7B-SFT": "SFT",
    "allenai/OLMo-2-1124-7B-DPO": "DPO",
    "allenai/OLMo-2-1124-7B-Instruct": "Instruct",
}
VARIANT_LINESTYLES = {"temperature": "-", "retok": "--"}
VARIANT_MARKERS = {"temperature": "o", "retok": "s"}
VARIANT_LABELS = {"temperature": "pass@k", "retok": "pass@retok"}

checkpoint_color_positions = np.linspace(0.25, 1, len(CHECKPOINT_MODELS))
checkpoint_colors = {model_name: plt.cm.viridis(checkpoint_color_positions[idx]) for idx, model_name in enumerate(CHECKPOINT_MODELS)}
posttrain_colors = {model_name: color for model_name, color in zip(POSTTRAIN_MODELS, ['#fca31d', '#e65c00', '#b30000', '#4a0000'])}
MODEL_COLORS = checkpoint_colors | posttrain_colors


In [ ]:
def requested_numvariants(model_name: str) -> int:
    if model_name.startswith("allenai_OLMo-2-1124-7B_step_"):
        return 51 if model_name.endswith("_1000") else 11
    return 51

def load_family(model_name: str, variant: str) -> pd.DataFrame:
    return eval_load.load_humaneval(
        model_name=model_name,
        dataset_size=DATASET_SIZE,
        numvariants=requested_numvariants(model_name),
        variant_type=variant,
    )

curve_data = {}
for model_name in LOAD_MODELS:
    curve_data[model_name] = {}
    for variant in VARIANTS:
        df = load_family(model_name, variant)
        curve_data[model_name][variant] = df

summary_rows = []
for model_name in ORDERED_MODELS:
    for variant in VARIANTS:
        df = curve_data[model_name][variant]
        summary = passat.summarize_task_outcomes(df)
        q_values = (summary["num_samples"] - summary["num_correct"]) / summary["num_samples"]
        summary_rows.append({
            "model_name": model_name,
            "variant": variant,
            "q_mean": float(np.mean(q_values)),
            "empirical_mass_B": float(np.mean(q_values > HIGH_TAIL_THRESHOLD)),
        })
summary_df = pd.DataFrame(summary_rows)
summary_df["model_order"] = summary_df["model_name"].map(ORDERED_MODELS.index)
summary_df = summary_df.sort_values(["model_order", "variant"]).reset_index(drop=True)

size_data = {
    model_name: {variant: curve_data[model_name][variant] for variant in VARIANTS}
    for model_name in SIZE_MODELS
}
checkpoint_data = {
    model_name: {variant: curve_data[model_name][variant] for variant in VARIANTS}
    for model_name in CHECKPOINT_MODELS
}


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for model_name in SIZE_MODELS:
    color = plt.cm.plasma(SIZE_MODELS.index(model_name) / len(SIZE_MODELS))
    for variant in VARIANTS:
        x, y, yerr = passat.pass_curve_points(size_data[model_name][variant], max_k=50)
        ax.fill_between(x, y - yerr, y + yerr, color=color, alpha=0.05)
        ax.plot(x, y, color=color, linestyle=VARIANT_LINESTYLES[variant])
size_handles = [ax.plot([], [], color=plt.cm.plasma(i / len(SIZE_MODELS)), label='-'.join(model_name.split('-')[-2:]))[0] for i, model_name in enumerate(SIZE_MODELS)]
metric_handles = [ax.plot([], [], color='black', linestyle=VARIANT_LINESTYLES[variant], label=VARIANT_LABELS[variant])[0] for variant in VARIANTS]
legend_1 = ax.legend(handles=size_handles, bbox_to_anchor=(1.35, 0.4), loc='upper right')
ax.add_artist(legend_1)
ax.legend(handles=metric_handles, bbox_to_anchor=(1.35, 0.75), loc='upper right')
ax.set_xlim(1, 50)
ax.set_xlabel('k')
ax.set_ylabel('Pass Rate')
ax.grid(True, alpha=0.3)
fig.savefig(FIGURE_DIR / 'params_olmo2_passat.svg', bbox_inches='tight')
fig.savefig(FIGURE_DIR / 'params_olmo2_passat.png', bbox_inches='tight')

fig, ax = plt.subplots(figsize=(7, 5))
for model_name in CHECKPOINT_MODELS:
    color = plt.cm.viridis(CHECKPOINT_MODELS.index(model_name) / len(CHECKPOINT_MODELS))
    for variant in VARIANTS:
        x, y, yerr = passat.pass_curve_points(checkpoint_data[model_name][variant], max_k=50)
        ax.fill_between(x, y - yerr, y + yerr, color=color, alpha=0.05)
        ax.plot(x, y, color=color, linestyle=VARIANT_LINESTYLES[variant])
checkpoint_handles = [ax.plot([], [], color=plt.cm.viridis(i / len(CHECKPOINT_MODELS)), label='-'.join(model_name.split('_')[-2:]))[0] for i, model_name in enumerate(CHECKPOINT_MODELS)]
metric_handles = [ax.plot([], [], color='black', linestyle=VARIANT_LINESTYLES[variant], label=VARIANT_LABELS[variant])[0] for variant in VARIANTS]
legend_1 = ax.legend(handles=checkpoint_handles, bbox_to_anchor=(1.35, 0.4), loc='upper right')
ax.add_artist(legend_1)
ax.legend(handles=metric_handles, bbox_to_anchor=(1.35, 0.75), loc='upper right')
ax.set_xlim(1, 50)
ax.set_xlabel('k')
ax.set_ylabel('Pass Rate')
ax.grid(True, alpha=0.3)
fig.savefig(FIGURE_DIR / 'checkpoints_olmo2_passat.svg', bbox_inches='tight')
fig.savefig(FIGURE_DIR / 'checkpoints_olmo2_passat.png', bbox_inches='tight')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)
ax_checkpoint, ax_size = axes
for model_name in CHECKPOINT_MODELS:
    color = MODEL_COLORS[model_name]
    for variant in VARIANTS:
        x, y, yerr = passat.pass_curve_points(checkpoint_data[model_name][variant], max_k=50)
        ax_checkpoint.fill_between(x, y - yerr, y + yerr, color=color, alpha=0.05)
        ax_checkpoint.plot(x, y, color=color, linestyle=VARIANT_LINESTYLES[variant])
for model_name in SIZE_MODELS:
    color = plt.cm.viridis(SIZE_MODELS.index(model_name) / len(SIZE_MODELS))
    for variant in VARIANTS:
        x, y, yerr = passat.pass_curve_points(size_data[model_name][variant], max_k=50)
        ax_size.fill_between(x, y - yerr, y + yerr, color=color, alpha=0.05)
        ax_size.plot(x, y, color=color, linestyle=VARIANT_LINESTYLES[variant])
ax_checkpoint.set_title('allenai_OLMo-2-1124-7B')
ax_size.set_title('OLMo 2 Model Sizes')
for ax in axes:
    ax.set_xlim(1, 50)
    ax.set_xlabel('k')
    ax.set_ylabel('Pass Rate')
    ax.grid(True, alpha=0.3)
fig.savefig(FIGURE_DIR / 'olmo2_combined_passat.svg', bbox_inches='tight')
fig.savefig(FIGURE_DIR / 'olmo2_combined_passat.png', bbox_inches='tight')

plot_models = ORDERED_MODELS
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
ax_curve, ax_tail = axes
for model_name in plot_models:
    color = MODEL_COLORS[model_name]
    for variant in VARIANTS:
        x, y, yerr = passat.pass_curve_points(curve_data[model_name][variant], max_k=50)
        ax_curve.fill_between(x, y - yerr, y + yerr, color=color, alpha=0.05)
        ax_curve.plot(x, y, color=color, linestyle=VARIANT_LINESTYLES[variant], linewidth=1.5)
ax_curve.legend(handles=[Line2D([0], [0], color='black', linestyle=VARIANT_LINESTYLES[variant], linewidth=2, label=VARIANT_LABELS[variant]) for variant in VARIANTS], title='Metric', loc='upper left', frameon=True)
ax_curve.set_xlim(1, 50)
ax_curve.set_ylim(-0.02, 1.02)
ax_curve.set_xlabel('k')
ax_curve.set_ylabel('Pass Rate')
ax_curve.grid(True, alpha=0.3)
checkpoint_steps = {
    'allenai_OLMo-2-1124-7B_step_1000': 1_000,
    'allenai_OLMo-2-1124-7B_step_10000': 10_000,
    'allenai_OLMo-2-1124-7B_step_30000': 30_000,
    'allenai_OLMo-2-1124-7B_step_60000': 60_000,
    'allenai_OLMo-2-1124-7B_step_101000': 101_000,
    'allenai_OLMo-2-1124-7B_step_300000': 300_000,
    'allenai_OLMo-2-1124-7B_step_500000': 500_000,
    'allenai_OLMo-2-1124-7B_step_700000': 700_000,
    'allenai_OLMo-2-1124-7B_step_900000': 900_000,
}
posttrain_log_step = 0.3
last_checkpoint_log10 = np.log10(checkpoint_steps[CHECKPOINT_MODELS[-1]])
posttrain_positions = {model_name: 10 ** (last_checkpoint_log10 + posttrain_log_step * (idx + 1)) for idx, model_name in enumerate(POSTTRAIN_MODELS)}
model_x_positions = checkpoint_steps | posttrain_positions
for variant in VARIANTS:
    subset = summary_df[summary_df['variant'] == variant].sort_values('model_order').reset_index(drop=True)
    x_values = np.array([model_x_positions[model_name] for model_name in subset['model_name']], dtype=float)
    y_values = subset['q_mean'].to_numpy(dtype=float)
    ax_tail.plot(x_values, y_values, color='black', linestyle=VARIANT_LINESTYLES[variant], linewidth=1.5, alpha=0.7)
    for xpos, row in zip(x_values, subset.itertuples(index=False)):
        ax_tail.scatter(xpos, row.q_mean, s=110, color=MODEL_COLORS[row.model_name], marker=VARIANT_MARKERS[variant], edgecolors='black', linewidths=0.8, alpha=0.95)
ax_tail.set_xlabel('Pre and post-training checkpoints')
ax_tail.set_ylabel(r'$\langle P_{fail}\rangle$')
ax_tail.set_xscale('log')
ax_tail.set_xlim(900, posttrain_positions[POSTTRAIN_MODELS[-1]] * 1.1)
ax_tail.set_xticks([model_x_positions[model_name] for model_name in plot_models])
ax_tail.set_xticklabels([MODEL_LABELS[model_name] for model_name in plot_models], rotation=45, ha='right')
ax_tail.grid(True, which='both', axis='y', alpha=0.3)
ax_tail.grid(True, which='major', axis='x', alpha=0.2)
fig.savefig(FIGURE_DIR / 'pre_post_humaneval_pass_curves_and_mean_failure_probabilities.png', bbox_inches='tight')
fig.savefig(FIGURE_DIR / 'pre_post_humaneval_pass_curves_and_mean_failure_probabilities.svg', bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for model_name in SIZE_MODELS:
    color = plt.cm.plasma(SIZE_MODELS.index(model_name) / len(SIZE_MODELS))
    x_temp, y_temp, _ = passat.pass_curve_points(size_data[model_name]["temperature"], max_k=50)
    x_retok, y_retok, _ = passat.pass_curve_points(size_data[model_name]["retok"], max_k=50)
    if not np.array_equal(x_temp, x_retok):
        raise ValueError(f"Mismatched k values for {model_name}")
    ax.plot(x_temp, y_temp - y_retok, color=color, linewidth=2, label='-'.join(model_name.split('-')[-2:]))
ax.axhline(0, color='black', linewidth=1, linestyle=':')
ax.set_xlim(1, 50)
ax.set_xlabel('k')
ax.set_ylabel('pass@k - pass@retok')
ax.grid(True, alpha=0.3)
ax.legend(title='Model Size')
fig.savefig(FIGURE_DIR / 'olmo2_passat_minus_retok_by_size.svg', bbox_inches='tight')
fig.savefig(FIGURE_DIR / 'olmo2_passat_minus_retok_by_size.png', bbox_inches='tight')


In [ ]:
RETOK_K = 1

plot_models = ORDERED_MODELS
retok_pass_at_k = []
for model_name in plot_models:
    x, y, yerr = passat.pass_curve_points(curve_data[model_name]["retok"], max_k=RETOK_K)
    k_matches = np.flatnonzero(x == RETOK_K)
    if len(k_matches) == 0:
        raise ValueError(f"k={RETOK_K} is not available for {model_name}; available k values span {x.min()}-{x.max()}")
    retok_pass_at_k.append((model_name, y[k_matches[0]], yerr[k_matches[0]]))

x_values = np.array([model_x_positions[model_name] for model_name, _, _ in retok_pass_at_k], dtype=float)
y_values = np.array([pass_rate for _, pass_rate, _ in retok_pass_at_k], dtype=float)
yerr_values = np.array([pass_rate_std for _, _, pass_rate_std in retok_pass_at_k], dtype=float)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x_values, y_values, color='black', linewidth=1.5, alpha=0.7)
for xpos, (model_name, pass_rate, pass_rate_std) in zip(x_values, retok_pass_at_k):
    ax.errorbar(
        xpos,
        pass_rate,
        yerr=pass_rate_std,
        fmt=VARIANT_MARKERS["retok"],
        markersize=9,
        color=MODEL_COLORS[model_name],
        markeredgecolor='black',
        markeredgewidth=0.8,
        ecolor=MODEL_COLORS[model_name],
        elinewidth=1.2,
        capsize=3,
        alpha=0.95,
    )

ax.set_xlabel('Pre and post-training checkpoints')
ax.set_ylabel(f'pass@retok at k={RETOK_K}')
ax.set_xscale('log')
ax.set_xlim(900, posttrain_positions[POSTTRAIN_MODELS[-1]] * 1.1)
ax.set_xticks([model_x_positions[model_name] for model_name in plot_models])
ax.set_xticklabels([MODEL_LABELS[model_name] for model_name in plot_models], rotation=45, ha='right')
ax.set_ylim(-0.02, 1.02)
ax.grid(True, which='both', axis='y', alpha=0.3)
ax.grid(True, which='major', axis='x', alpha=0.2)
fig.savefig(FIGURE_DIR / f'pre_post_humaneval_pass_retok_at_k{RETOK_K}.png', bbox_inches='tight')
fig.savefig(FIGURE_DIR / f'pre_post_humaneval_pass_retok_at_k{RETOK_K}.svg', bbox_inches='tight')


In [ ]:
retok_pass_at_k

In [ ]:
RETOK_K_VALUES = [1, 20, 45]

plot_models = ORDERED_MODELS
fig, ax = plt.subplots(figsize=(7, 5))
for retok_k in RETOK_K_VALUES:
    retok_pass_at_k = []
    for model_name in plot_models:
        x, y, yerr = passat.pass_curve_points(curve_data[model_name]["retok"], max_k=retok_k)
        k_matches = np.flatnonzero(x == retok_k)
        if len(k_matches) == 0:
            raise ValueError(f"k={retok_k} is not available for {model_name}; available k values span {x.min()}-{x.max()}")
        retok_pass_at_k.append((model_name, y[k_matches[0]], yerr[k_matches[0]]))

    x_values = np.array([model_x_positions[model_name] for model_name, _, _ in retok_pass_at_k], dtype=float)
    y_values = np.array([pass_rate for _, pass_rate, _ in retok_pass_at_k], dtype=float)
    yerr_values = np.array([pass_rate_std for _, _, pass_rate_std in retok_pass_at_k], dtype=float)

    ax.errorbar(
        x_values,
        y_values,
        yerr=yerr_values,
        marker='o',
        linewidth=1.5,
        markersize=5,
        capsize=3,
        label=f'k={retok_k}',
    )

ax.set_xlabel('Pre and post-training checkpoints')
ax.set_ylabel('pass@retok')
ax.set_xscale('log')
ax.set_xlim(900, posttrain_positions[POSTTRAIN_MODELS[-1]] * 1.1)
ax.set_xticks([model_x_positions[model_name] for model_name in plot_models])
ax.set_xticklabels([MODEL_LABELS[model_name] for model_name in plot_models], rotation=45, ha='right')
ax.set_ylim(-0.02, 1.02)
ax.grid(True, which='both', axis='y', alpha=0.3)
ax.grid(True, which='major', axis='x', alpha=0.2)
ax.legend(title='pass@retok')
